In [2]:
using CSV, DataFrames, JSON, CodecZlib

# Read the first line and process it
metadata_string = readline(GzipDecompressorStream(open("Data/SCoV1_M6_Sort1_1_Paired_All.csv.gz")))
metadata_string = replace(metadata_string, "\"\"" => "\"")  # Replace double quotes with single quotes
metadata_string = strip(metadata_string, '"')  # Remove the outermost quotes
metadata = JSON.parse(metadata_string)

# Read the CSV data, skipping the first line
df = CSV.File("Data/SCoV1_M6_Sort1_1_Paired_All.csv.gz", 
              skipto=3,  # Skip the metadata line
              header=2,  # Use the second line as header
              delim=',', 
              missingstring=["", "NA"],
              ignoreemptyrows=true,
              silencewarnings=true) |> DataFrame;

;
# Now you have both metadata and the DataFrame

In [3]:
@show sequence = [i for i in names(df) if occursin("aa", i) && occursin("sequence", i)];

sequence = [i for i = names(df) if occursin("aa", i) && occursin("sequence", i)] = ["sequence_alignment_aa_heavy", "v_sequence_alignment_aa_heavy", "d_sequence_alignment_aa_heavy", "j_sequence_alignment_aa_heavy", "sequence_alignment_aa_light", "v_sequence_alignment_aa_light", "d_sequence_alignment_aa_light", "j_sequence_alignment_aa_light"]


In [4]:
for name in names(df)
    println(name)
end

sequence_id_heavy
sequence_heavy
locus_heavy
stop_codon_heavy
vj_in_frame_heavy
v_frameshift_heavy
productive_heavy
rev_comp_heavy
complete_vdj_heavy
v_call_heavy
d_call_heavy
j_call_heavy
sequence_alignment_heavy
germline_alignment_heavy
sequence_alignment_aa_heavy
germline_alignment_aa_heavy
v_alignment_start_heavy
v_alignment_end_heavy
d_alignment_start_heavy
d_alignment_end_heavy
j_alignment_start_heavy
j_alignment_end_heavy
v_sequence_alignment_heavy
v_sequence_alignment_aa_heavy
v_germline_alignment_heavy
v_germline_alignment_aa_heavy
d_sequence_alignment_heavy
d_sequence_alignment_aa_heavy
d_germline_alignment_heavy
d_germline_alignment_aa_heavy
j_sequence_alignment_heavy
j_sequence_alignment_aa_heavy
j_germline_alignment_heavy
j_germline_alignment_aa_heavy
fwr1_heavy
fwr1_aa_heavy
cdr1_heavy
cdr1_aa_heavy
fwr2_heavy
fwr2_aa_heavy
cdr2_heavy
cdr2_aa_heavy
fwr3_heavy
fwr3_aa_heavy
fwr4_heavy
fwr4_aa_heavy
cdr3_heavy
cdr3_aa_heavy
junction_heavy
junction_length_heavy
junction_aa_h

# **Understanding the DataFrame**

As can be seen above, the data consists of *a lot* of different columns. Some of these consists of DNA/AA sequences whereas others describe the functionality/quality of the antibody. Here is a slight overview of what is taken into account in the following processing code. Source: Claude.ai

## What is consists of

### Quality and validity indicators:

- productive_heavy: Indicates if the sequence is productive (can potentially produce a functional antibody).
- stop_codon_heavy: Presence of premature stop codons.
- vj_in_frame_heavy: Whether the V and J genes are in-frame.
- v_frameshift_heavy: Presence of frameshifts in the V gene.
- complete_vdj_heavy: Whether the sequence has complete V, D, and J gene segments.


### Gene calls:

- v_call_heavy, d_call_heavy, j_call_heavy: The identified V, D, and J genes.


### Alignment scores and identity:

- v_score_heavy, d_score_heavy, j_score_heavy: Alignment scores for V, D, and J genes.
- v_identity_heavy, d_identity_heavy, j_identity_heavy: Percent identity to germline genes.


### CDR3 information:

- cdr3_aa_heavy: The amino acid sequence of the CDR3 region.
- cdr3_heavy: The nucleotide sequence of the CDR3 region.
- junction_length_heavy, junction_aa_length_heavy: Lengths of the junction region.


### Isotype information:

- Isotype_heavy: The antibody isotype.


### Sequence redundancy:

- Redundancy_heavy: Indicates how many times this sequence was observed.


### ANARCI information:

- ANARCI_numbering_heavy, ANARCI_status_heavy: Numbering and status from the ANARCI tool, which can provide additional structural information.

## Concluding remarks

For your diffusion task, you might want to consider filtering or weighting sequences based on some of these quality indicators. For example, you might only want to use sequences that are productive, have complete VDJ regions, and don't have stop codons or frameshifts.
Additionally, the CDR3 region (especially cdr3_aa_heavy) is often of particular interest in antibody studies as it's crucial for antigen binding.
Remember that this dataset includes both heavy and light chain information (indicated by '_heavy' and '_light' suffixes). Depending on your specific task, you might want to consider both chains or focus on one.



In [7]:
sequences = df.sequence_alignment_aa_heavy

7979-element Vector{String}:
 "QVQVVHSGAEVKRPGSSVKVSCKTSGYPFSS" ⋯ 57 bytes ⋯ "EDTAVYFCAREIPATSYFDYWGQGTLVTVSS"
 "EVQLVESGGGLVKPGGSLRLSCVGSNFPFKD" ⋯ 68 bytes ⋯ "TDRAEGFYSDVKGYTSEGHFWGQGTLVTVSS"
 "QVQLQESGPGLVKPSETLSLTCTVSGGSMSR" ⋯ 56 bytes ⋯ "ADTAVYYCARVNYYESYFEYWGQGSLLTVSS"
 "QMQLVESGGGVVQPGKSLRLSCAASGFTFYN" ⋯ 63 bytes ⋯ "YCAKVVTPYDYADPDNWLDPWGQGTLVTVSS"
 "QVQLQESGPGLVKPSQTLSLTCTVSGGSISS" ⋯ 63 bytes ⋯ "YYCASGSYYDILVPPYGMDVWGQGTTVTVSS"
 "EVQLLESGGGLVQPGGSLRLSCAASGFTFSS" ⋯ 63 bytes ⋯ "YCAKPPGRFWSGYYGCGMDVWGQGTTVTVSS"
 "EVHLVQSGGEVKKPGESLRISCQASGYSFTN" ⋯ 60 bytes ⋯ "AKYYCARLIQTGAAAGTFDNWGQGTLVTVSS"
 "QVQLVQSGAEVKKPGASVKVSCKTSGYTLTN" ⋯ 55 bytes ⋯ "ISDDTAVYFCATVYSHGPETWGQGTLVTVSS"
 "DVQLLESGGGLVQPGGSLRLSCAASGFTFAS" ⋯ 65 bytes ⋯ "ARWTVAGRGADFYYSYGMDVWGQGTTVTVSS"
 "QVQLVESGGGVVQPGRSLRLSCATSGFTFKN" ⋯ 60 bytes ⋯ "AVYYCARDQNSGSYPGALDYWGQGTLVTVSP"
 "QVQLVQSGAEVKKPGASVKVSCKASGYTFTG" ⋯ 58 bytes ⋯ "DTAVYYCARTSSWYHYGMDVWGQGTTVTVSS"
 "VQLVESGGGLVEPGGSLRLSCAASGFRFHTA" ⋯ 55 bytes ⋯ "LKTDDTAVYYCATDRGDLDN

# Processing the Data